In [1]:
# NEW NOTEBOOK: CELL 1  Load dataset

import torch
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

load_path = "gnn_structural_netlist_dataset.pt"  # adjust path if needed

obj = torch.load(load_path, map_location="cpu")

data_list = obj["data_list"]
all_gate_types = obj["all_gate_types"]
gate_type_to_idx = obj["gate_type_to_idx"]

print("[INFO] Loaded dataset")
print("  #graphs        :", len(data_list))
print("  all_gate_types :", all_gate_types)
print("  feat_dim (x)   :", data_list[0].x.size(-1))
print("  example circuits:", [d.circuit_name for d in data_list[:5]])


/home/rrk307/anaconda3/envs/gnn_circuits/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/rrk307/anaconda3/envs/gnn_circuits/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/rrk307/anaconda3/envs/gnn_circuits/lib/python3.10/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /home/rrk307/anaconda3/envs/gnn_circuits/lib/python3.10/site-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
/home/rrk307/anaconda3/envs/gnn_circuits/lib/python3.10/site-packages/torch_geometric/typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-

[INFO] Loaded dataset
  #graphs        : 30
  all_gate_types : ['and', 'nand', 'nor', 'not', 'or', 'xor']
  feat_dim (x)   : 10
  example circuits: ['c17.v', 'c432.v', 'c499.v', 'ctrl.v', 'c880.v']


/tmp/ipykernel_35357/2234914924.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(load_path, map_location="cpu")


In [2]:
# NEW NOTEBOOK: CELL 2  Per-circuit normalization of M(v)

print("[INFO] Per-circuit normalization of M(v) on gate nodes.")

for d in data_list:
    gate_mask = d.gate_mask
    y_gate = d.y[gate_mask]  # (num_gates, 1)

    if y_gate.numel() == 0:
        d.y_norm = d.y.clone()
        d.y_mean = torch.zeros_like(d.y)
        d.y_std = torch.ones_like(d.y)
        continue

    mean = y_gate.mean().item()
    std = y_gate.std().item()
    if std < 1e-6:
        std = 1.0

    d.y_mean = torch.full_like(d.y, mean)
    d.y_std = torch.full_like(d.y, std)
    d.y_norm = (d.y - mean) / std

print("[INFO] Normalization complete.")


[INFO] Per-circuit normalization of M(v) on gate nodes.
[INFO] Normalization complete.


In [3]:
# CELL 3: HetGNN-style training & evaluation with multi-seed, baseline, per-circuit metrics

import random
import numpy as np
import torch
from torch import nn
from torch_geometric.nn import MessagePassing
from torch_geometric.loader import DataLoader
import networkx as nx

# -------------------------------------------------------------------
# 1) Spearman correlation (numpy)
# -------------------------------------------------------------------
def spearmanr_np(y_true, y_pred):
    """
    Simple Spearman rank correlation using numpy.
    Returns scalar in [-1, 1] (or NaN if undefined).
    """
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    if y_true.size < 2:
        return np.nan

    def rankdata(a):
        order = np.argsort(a)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(len(a))
        return ranks

    r1 = rankdata(y_true)
    r2 = rankdata(y_pred)

    r1_mean = r1.mean()
    r2_mean = r2.mean()
    num = np.sum((r1 - r1_mean) * (r2 - r2_mean))
    den = np.sqrt(np.sum((r1 - r1_mean) ** 2) * np.sum((r2 - r2_mean) ** 2))
    if den == 0:
        return np.nan
    return num / den


# -------------------------------------------------------------------
# 2) Build node_type for each graph: 0=gate, 1=input, 2=output
# -------------------------------------------------------------------
print("[INFO] Building node_type (0=gate, 1=input, 2=output) for each circuit...")

num_gate_types = len(all_gate_types)  # from Cell 1

for d in data_list:
    x = d.x  # [N, F]
    # By construction: x[..., num_gate_types]   = is_input
    #                  x[..., num_gate_types+1] = is_output
    is_input = x[:, num_gate_types] > 0.5
    is_output = x[:, num_gate_types + 1] > 0.5
    # default = gate
    node_type = torch.zeros(x.size(0), dtype=torch.long)
    node_type[is_input] = 1
    node_type[is_output] = 2
    d.node_type = node_type

print("[INFO] node_type added to all graphs.")


# -------------------------------------------------------------------
# 3) HetGNN-like layer & regressor (type-aware message passing)
# -------------------------------------------------------------------
class HetGNNLayer(MessagePassing):
    def __init__(self, in_dim, type_emb_dim, out_dim, num_types=3):
        super().__init__(aggr='add')  # sum aggregation
        self.type_emb = nn.Embedding(num_types, type_emb_dim)

        self.msg_mlp = nn.Sequential(
            nn.Linear(in_dim + type_emb_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

        self.update_mlp = nn.Sequential(
            nn.Linear(in_dim + type_emb_dim + out_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, x, edge_index, node_type):
        # x: [N, in_dim], node_type: [N]
        t_emb = self.type_emb(node_type)             # [N, type_emb_dim]
        x_cat = torch.cat([x, t_emb], dim=-1)        # [N, in_dim + type_emb_dim]
        m = self.propagate(edge_index, x_cat=x_cat)  # [N, out_dim]
        h_cat = torch.cat([x_cat, m], dim=-1)        # [N, in_dim + type_emb_dim + out_dim]
        return self.update_mlp(h_cat)                # [N, out_dim]

    def message(self, x_cat_j):
        # x_cat_j: [E, in_dim + type_emb_dim] from neighbors
        return self.msg_mlp(x_cat_j)


class HetGNNRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, type_emb_dim=8, num_layers=3, num_types=3):
        super().__init__()
        self.layers = nn.ModuleList()

        # First layer: in_dim -> hidden_dim
        self.layers.append(HetGNNLayer(in_dim, type_emb_dim, hidden_dim, num_types=num_types))
        # Hidden layers: hidden_dim -> hidden_dim
        for _ in range(num_layers - 1):
            self.layers.append(HetGNNLayer(hidden_dim, type_emb_dim, hidden_dim, num_types=num_types))

        self.act = nn.ReLU()
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, node_type):
        h = x
        for layer in self.layers:
            h = layer(h, edge_index, node_type)
            h = self.act(h)
        return self.out(h)


# -------------------------------------------------------------------
# 4) Precompute baseline: 1 - betweenness centrality (approximate, once per graph)
# -------------------------------------------------------------------
print("[INFO] Precomputing baseline (1 - betweenness) per circuit...")

for d in data_list:
    if hasattr(d, "baseline_scores"):
        continue  # already computed in some earlier experiment

    num_nodes = d.num_nodes
    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(num_nodes))

    edge_index = d.edge_index.cpu().numpy()
    for u, v in edge_index.T:
        G_nx.add_edge(int(u), int(v))

    if num_nodes < 3:
        bc = {v: 0.0 for v in G_nx.nodes()}
    else:
        # approximate betweenness for speed
        k = min(128, num_nodes)
        try:
            bc = nx.betweenness_centrality(G_nx, normalized=True, k=k)
        except ZeroDivisionError:
            bc = {v: 0.0 for v in G_nx.nodes()}

    baseline = np.zeros(num_nodes, dtype=np.float32)
    for v in G_nx.nodes():
        baseline[v] = 1.0 - float(bc.get(v, 0.0))

    d.baseline_scores = torch.from_numpy(baseline)

print("[INFO] Baseline precomputation done.")


# -------------------------------------------------------------------
# 5) Multi-seed evaluation for HetGNN-like model
# -------------------------------------------------------------------
if len(data_list) == 0:
    raise RuntimeError("data_list is empty. Make sure load + normalization cells ran correctly.")

device = "cuda" if torch.cuda.is_available() else "cpu"
in_dim = data_list[0].x.size(-1)

print(f"[INFO] Using device: {device}")
print(f"[INFO] Input feature dimension: {in_dim}")

seeds = [0, 1]   # keep small for speed
num_epochs = 10  # fast experiments

results_gnn = []
results_base = []

for seed in seeds:
    print(f"\n================ HetGNN  SEED {seed} ================")

    # ---- train/test split by circuit for this seed ----
    indices = list(range(len(data_list)))
    rnd = random.Random(seed)
    rnd.shuffle(indices)

    split = int(0.8 * len(indices))
    train_indices = indices[:split]
    test_indices = indices[split:]

    train_dataset = [data_list[i] for i in train_indices]
    test_dataset = [data_list[i] for i in test_indices]

    print("[SPLIT] Train circuits:", [d.circuit_name for d in train_dataset])
    print("[SPLIT] Test circuits :", [d.circuit_name for d in test_dataset])

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

    # ---- model & optimizer ----
    model = HetGNNRegressor(in_dim=in_dim, hidden_dim=64, type_emb_dim=8, num_layers=3, num_types=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # ---- training (on normalized targets) ----
    model.train()
    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        total_gates = 0

        for batch in train_loader:
            batch = batch.to(device)
            pred_norm = model(batch.x, batch.edge_index, batch.node_type)  # (N, 1)
            gate_mask = batch.gate_mask

            pred_gate_norm = pred_norm[gate_mask]
            y_gate_norm = batch.y_norm[gate_mask]

            if pred_gate_norm.numel() == 0:
                continue

            loss = loss_fn(pred_gate_norm, y_gate_norm)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            num_gates = gate_mask.sum().item()
            total_loss += loss.item() * num_gates
            total_gates += num_gates

        avg_loss = total_loss / max(1, total_gates)
        print(f"[SEED {seed}] Epoch {epoch:03d} | Train MSE (normalized, gate nodes): {avg_loss:.6f}")

    # ---- evaluation on test circuits ----
    model.eval()
    all_true = []
    all_pred = []
    all_base = []

    per_circuit_spearman = []
    per_circuit_base_spearman = []

    with torch.no_grad():
        for data in test_dataset:
            d = data.to(device)
            pred_norm = model(d.x, d.edge_index, d.node_type)  # (N, 1)

            # unnormalize predictions back to original M(v) scale
            pred = (pred_norm * d.y_std + d.y_mean).cpu().numpy().flatten()
            y_true = d.y.cpu().numpy().flatten()
            gate_mask = d.gate_mask.cpu().numpy().astype(bool)

            pred_gate = pred[gate_mask]
            y_gate = y_true[gate_mask]

            if pred_gate.size == 0:
                continue

            # collect for global metrics
            all_true.append(y_gate)
            all_pred.append(pred_gate)

            # baseline scores
            base_scores = data.baseline_scores.cpu().numpy().flatten()
            base_gate = base_scores[gate_mask]
            all_base.append(base_gate)

            # per-circuit Spearman
            s_gnn = spearmanr_np(y_gate, pred_gate)
            s_base = spearmanr_np(y_gate, base_gate)
            per_circuit_spearman.append((data.circuit_name, s_gnn))
            per_circuit_base_spearman.append((data.circuit_name, s_base))

    if len(all_true) == 0:
        print("[WARN] No gate nodes in test set for this seed.")
        continue

    all_true = np.concatenate(all_true, axis=0)
    all_pred = np.concatenate(all_pred, axis=0)
    all_base = np.concatenate(all_base, axis=0)

    # Global metrics
    mse_gnn = np.mean((all_true - all_pred) ** 2)
    mae_gnn = np.mean(np.abs(all_true - all_pred))
    spearman_gnn = spearmanr_np(all_true, all_pred)

    mse_base = np.mean((all_true - all_base) ** 2)
    mae_base = np.mean(np.abs(all_true - all_base))
    spearman_base = spearmanr_np(all_true, all_base)

    results_gnn.append((mse_gnn, mae_gnn, spearman_gnn))
    results_base.append((mse_base, mae_base, spearman_base))

    print("\n[SEED {} TEST RESULTS] HetGNN-like (gate nodes, unseen circuits)".format(seed))
    print(f"  MSE      : {mse_gnn:.6f}")
    print(f"  MAE      : {mae_gnn:.6f}")
    print(f"  Spearman : {spearman_gnn:.4f}")

    print("\n[SEED {} TEST RESULTS] Baseline (1 - betweenness)".format(seed))
    print(f"  MSE      : {mse_base:.6f}")
    print(f"  MAE      : {mae_base:.6f}")
    print(f"  Spearman : {spearman_base:.4f}")

    # Per-circuit Spearman
    print("\n[SEED {}] Per-circuit Spearman (HetGNN-like vs baseline)".format(seed))
    for (cname, s_g), (_, s_b) in zip(per_circuit_spearman, per_circuit_base_spearman):
        print(f"  {cname:15s}  GNN={s_g:+.3f}  Base={s_b:+.3f}")


# -------------------------------------------------------------------
# 6) Aggregate results over seeds
# -------------------------------------------------------------------
def summarize_results(results):
    arr = np.array(results)  # shape (seeds, 3)
    mse_mean, mae_mean, sp_mean = arr.mean(axis=0)
    mse_std, mae_std, sp_std = arr.std(axis=0)
    return (mse_mean, mse_std), (mae_mean, mae_std), (sp_mean, sp_std)

if len(results_gnn) > 0:
    (mse_m, mse_s), (mae_m, mae_s), (sp_m, sp_s) = summarize_results(results_gnn)
    print("\n================ AGGREGATE HetGNN-like RESULTS OVER SEEDS ================")
    print(f"  MSE      : {mse_m:.6f} ± {mse_s:.6f}")
    print(f"  MAE      : {mae_m:.6f} ± {mae_s:.6f}")
    print(f"  Spearman : {sp_m:.4f} ± {sp_s:.4f}")

if len(results_base) > 0:
    (mse_m, mse_s), (mae_m, mae_s), (sp_m, sp_s) = summarize_results(results_base)
    print("\n================ AGGREGATE BASELINE RESULTS OVER SEEDS ================")
    print(f"  MSE      : {mse_m:.6f} ± {mse_s:.6f}")
    print(f"  MAE      : {mae_m:.6f} ± {mae_s:.6f}")
    print(f"  Spearman : {sp_m:.4f} ± {sp_s:.4f}")


[INFO] Building node_type (0=gate, 1=input, 2=output) for each circuit...
[INFO] node_type added to all graphs.
[INFO] Precomputing baseline (1 - betweenness) per circuit...
[INFO] Baseline precomputation done.
[INFO] Using device: cpu
[INFO] Input feature dimension: 10

================ HetGNN  SEED 0 ================
[SPLIT] Train circuits: ['ctrl.v', 'c2670.v', 'c7552.v', 'c6288.v', 'int2float.v', 'c17.v', 'c1908.v', 'multiplier.v', 'div.v', 'c5315.v', 'max.v', 'sqrt.v', 'sin.v', 'c499.v', 'bar.v', 'c880.v', 'voter.v', 'router.v', 'c3540.v', 'arbiter.v', 'dec.v', 'memctrl.v', 'i2c.v', 'adder.v']
[SPLIT] Test circuits : ['c1355.v', 'c432.v', 'Priority.v', 'square.v', 'cavlc.v', 'log2.v']
[SEED 0] Epoch 001 | Train MSE (normalized, gate nodes): 1.013065
[SEED 0] Epoch 002 | Train MSE (normalized, gate nodes): 0.995976
[SEED 0] Epoch 003 | Train MSE (normalized, gate nodes): 0.997020
[SEED 0] Epoch 004 | Train MSE (normalized, gate nodes): 0.990678
[SEED 0] Epoch 005 | Train MSE (norm